# Session 2: Strands Agents를 활용한 Agent 구축

### Index
1. LLM vs Agent — 개념 이해
2. Strands Agents 기본 사용
3. Tool 정의 및 활용
4. 실전 예제: 여러 Tool 조합
5. 단기 메모리(Short-term Memory) 확인
6. Session 관리 — 대화 분리 및 복원
7. Agent를 AgentCore에 배포하기

---
## 1. LLM vs Agent

### LLM (Session 1에서 한 것)
- 질문 → 답변 (1회성)
- 외부 정보 접근 불가
- 실시간 데이터 모름

### Agent
- 목표를 주면 **스스로 판단**하고 **도구(Tool)를 사용**해서 해결
- 필요하면 여러 단계를 거쳐 작업 수행
- ReAct 패턴: **생각(Reasoning) → 행동(Action) → 관찰(Observation)** 반복

```
사용자: "서울 날씨 알려주고 화씨로 변환해줘"

Agent 내부:
  [생각] 서울 날씨를 먼저 조회해야겠다 → get_weather 호출
  [행동] get_weather("서울") → "맑음, 23도"
  [관찰] 섭씨 23도를 화씨로 변환해야 한다
  [행동] calculator("23 * 9/5 + 32") → "73.4"
  [답변] 서울은 맑음이고 23°C(73.4°F)입니다.
```

> 💡 Agent = LLM + Tool + 판단 루프

---
## 2. Strands Agents 기본 사용

> [Strands Agents](https://github.com/strands-agents/sdk-python)는 AWS에서 만든 오픈소스 Agent 프레임워크입니다.  
> Bedrock 모델과 자연스럽게 연동되며, 간단한 코드로 Agent를 만들 수 있습니다.

In [4]:
# ! pip install strands-agents

In [6]:
from strands import Agent
from strands.models import BedrockModel

In [7]:
model = BedrockModel(
  model_id="us.amazon.nova-2-lite-v1:0",
  region_name="us-east-1"
)

### 기본 Agent 생성 및 대화

In [8]:
agent = Agent(model=model)
response = agent("안녕하세요! 당신은 누구인가요?")

안녕하세요! 저는 인공지능 어시스턴트입니다. 저를 만들어준 개발자들은 저를 대화 상대로, 정보 제공자로, 문제 해결 도우미로 설계했습니다. 

무엇을 도와드릴까요? 질문이 있으시면 언제든지 알려주세요!

### System Prompt로 역할 부여

> Agent에 성격과 역할을 부여하면 일관된 응답을 생성합니다.

In [10]:
agent = Agent(
  model=model,
  system_prompt="너는 AWS 전문 솔루션즈 아키텍트야. 모든 질문에 AWS 서비스를 활용한 답변을 해줘. 한국어로 10문장 이내로 답변해."
)
response = agent("사용자 인증 시스템을 만들고 싶어")

AWS에서 사용자 인증 시스템을 만들려면 **Amazon Cognito**를 사용합니다. **Amazon Cognito**는 사용자 풀을 생성하여 이메일, 전화번호 또는 소셜 미디어 계정을 통해 사용자를 관리할 수 있습니다. **Cognito**는 보안 강화 기능을 제공하며, 암호 정책과 멀티 팩터 인증(MFA)을 설정할 수 있습니다. 또한 API 게이트웨이와 연결하여 인증된 사용자만 접근할 수 있도록 제어할 수 있습니다. **IAM 역할**을 통해 AWS 리소스에 접근 권한을 부여할 수도 있습니다.

---
## 3. Tool 정의 및 활용

> Agent가 외부 세계와 상호작용하려면 **Tool(도구)**이 필요합니다.  
> Strands에서는 `@tool` 데코레이터로 간단하게 정의할 수 있습니다.

### Tool 작성 규칙
1. `@tool` 데코레이터 붙이기
2. **docstring 필수** — Agent가 이 설명을 보고 언제 사용할지 판단
3. 파라미터에 **타입 힌트** 필수 — Agent가 어떤 값을 넣을지 결정

In [11]:
from strands import Agent, tool
from strands.models import BedrockModel

In [12]:
@tool
def get_weather(city: str) -> str:
  """도시의 현재 날씨를 조회합니다. city는 도시 이름입니다."""
  # 실제로는 외부 API를 호출하겠지만, 여기서는 하드코딩
  weather_data = {
      "서울": "맑음, 23°C",
      "부산": "흐림, 19°C",
      "제주": "비, 21°C"
  }
  return weather_data.get(city, f"{city}의 날씨 정보를 찾을 수 없습니다.")

@tool
def calculator(expression: str) -> str:
  """수학 계산을 수행합니다. expression은 계산할 수식입니다. 예: '2 + 3', '100 * 0.15'"""
  try:
      result = eval(expression)
      return str(result)
  except Exception as e:
      return f"계산 오류: {e}"

### Tool을 Agent에 연결

In [13]:
model = BedrockModel(
  model_id="us.amazon.nova-2-lite-v1:0",
  region_name="us-east-1"
)

agent = Agent(
  model=model,
  tools=[get_weather, calculator],
  system_prompt="너는 친절한 비서야. 도구를 활용해서 정확한 정보를 제공해. 한국어로 답변해."
)

### Tool 호출 테스트

> Agent가 질문을 분석하고, 적절한 Tool을 선택하여 호출하는 과정을 관찰합니다.

In [14]:
# 날씨 Tool 호출
response = agent("서울 날씨 어때?")


Tool #1: get_weather
서울의 현재 날씨는 맑고 23°C입니다. 편안한 날씨네요!

In [15]:
# 계산 Tool 호출
response = agent("15% 팁 포함해서 45000원짜리 식사 총 금액은?")


Tool #2: calculator
15% 팁을 포함한 45000원짜리 식사의 총 금액은 약 **51,750원**입니다. (정확히는 51,750원이지만 소수점이 생길 수 있으므로 반올림해서 사용합니다)

In [16]:
# 여러 Tool 조합 호출
response = agent("서울 날씨 알려주고, 현재 온도를 화씨로 변환해줘")


Tool #3: get_weather

Tool #4: calculator
서울 날씨는 맑고, 현재 기온은 **23°C**입니다. 이를 화씨로 변환하면 **약 73.4°F**입니다.

---
## 4. 실전 예제: 여러 Tool 조합

> 좀 더 현실적인 시나리오로 Agent를 구성해봅니다.  
> **시나리오: 업무 비서** — 일정 조회, 계산, 현재 시간 확인이 가능한 Agent
>
> 💡 Strands Agent는 대화 히스토리(memory)를 자체 관리합니다.  
> 별도 메모 기능 없이도 이전 대화 내용을 기억합니다.

In [17]:
from strands import Agent, tool
from strands.models import BedrockModel
from datetime import datetime

@tool
def get_schedule(date: str) -> str:
  """특정 날짜의 일정을 조회합니다. date는 'YYYY-MM-DD' 형식입니다."""
  schedules = {
      "2026-05-05": "10:00 팀 미팅, 14:00 고객사 미팅, 16:00 코드 리뷰",
      "2026-05-06": "09:00 스프린트 플래닝, 15:00 1:1 미팅",
      "2026-05-07": "종일 워크샵"
  }
  return schedules.get(date, f"{date}에 등록된 일정이 없습니다.")

@tool
def get_current_time() -> str:
  """현재 날짜와 시간을 반환합니다."""
  return datetime.now().strftime("%Y-%m-%d %H:%M")

@tool
def calculator(expression: str) -> str:
  """수학 계산을 수행합니다. expression은 계산할 수식입니다."""
  try:
      return str(eval(expression))
  except Exception as e:
      return f"계산 오류: {e}"

In [18]:
model = BedrockModel(
  model_id="us.amazon.nova-2-lite-v1:0",
  region_name="us-east-1"
)

assistant = Agent(
  model=model,
  tools=[get_schedule, get_current_time, calculator],
  system_prompt="""너는 업무 비서 Agent야. 다음 도구를 활용해서 사용자를 도와줘:
- 일정 조회
- 현재 시간 확인
- 계산
한국어로 답변해."""
)

---
## 5. 단기 메모리(Short-term Memory) 확인

> Strands Agent는 **대화 히스토리를 자동으로 관리**합니다.  
> 같은 Agent 인스턴스에 여러 번 말을 걸면, 이전 대화 내용이 모두 LLM에 전달됩니다.  
> 이를 통해 Agent는 문맥을 유지하며 연속적인 대화가 가능합니다.
>
> 아래 실습에서 3번 연속 호출하면서:
> 1. `agent.messages` — 실제 LLM에 전달되는 메시지 배열 확인
> 2. 토큰 사용량 — 대화가 쌓일수록 input 토큰이 증가하는 것을 확인

In [1]:
from strands import Agent
from strands.models import BedrockModel

model = BedrockModel(
  model_id="us.amazon.nova-2-lite-v1:0",
  region_name="us-east-1"
)

agent = Agent(
  model=model,
  system_prompt="너는 친절한 비서야. 한국어로 간결하게 답변해."
)

print("=" * 50)
print("Agent 생성 완료. 대화 시작 전 메시지 수:", len(agent.messages))
print("=" * 50)

Agent 생성 완료. 대화 시작 전 메시지 수: 0


In [2]:
# 1번째 대화
response = agent("내 이름은 민수야. 기억해줘.")

print("\n" + "=" * 50)
print(f"[1번째 호출 후] 메시지 수: {len(agent.messages)}")
print("=" * 50)

물론입니다, 민수 씨! 이제 저는 여러분의 이름을 기억하겠습니다. 어떻게 도와드릴까요?
[1번째 호출 후] 메시지 수: 2


In [3]:
# 2번째 대화
response = agent("나는 서울에 살고 있어.")

print("\n" + "=" * 50)
print(f"[2번째 호출 후] 메시지 수: {len(agent.messages)}")
print("=" * 50)

알겠습니다, 민수 씨. 서울에 사시는군요! 서울에 관한 추가 정보가 필요하시거나, 도시 생활에 대한 팁, 관광 명소, 교통 정보 등이 필요하시다면 언제든지 물어보세요. 기꺼이 도와드리겠습니다. 

예를 들어, 민수 씨가 관심 있는 특정 지역이나 활동이 있다면 그에 대한 상세한 정보를 제공해드릴 수 있습니다. 또는 서울에서 자주 이용하는 교통수단에 대한 최신 정보도 알려드릴 수 있습니다.

무엇을 도와드릴까요?
[2번째 호출 후] 메시지 수: 4


In [4]:
# 3번째 대화 — 이전 대화 내용을 기억하는지 확인
response = agent("내 이름이 뭐라고 했지? 어디 산다고 했어?")

print("\n" + "=" * 50)
print(f"[3번째 호출 후] 메시지 수: {len(agent.messages)}")
print("=" * 50)

민수 씨는 서울에 사신다고 하셨습니다. 다시 한번 확인해 드리자면, 귀하의 이름은 **민수**이고, 거주지는 **서울**입니다. 

궁금한 것이 있으시면 언제든지 물어보세요!
[3번째 호출 후] 메시지 수: 6


In [7]:
# 이 셀을 여러 번 반복 실행해보세요 — inputTokens가 점점 증가합니다!
response = agent("안녕")
print(f"✅ inputTokens: {response.metrics.accumulated_usage['inputTokens']}")

안녕하세요, 민수 씨! 지금 어떤 도움이 필요하신가요? 언제든지 질문해 주시면 최선을 다해 도와드리겠습니다.✅ inputTokens: 1649


---
## 6. Session 관리 — 대화 분리 및 복원

> 단기 메모리는 같은 Agent 인스턴스 내에서만 유지됩니다.  
> 프로그램을 재시작하면 대화 히스토리가 사라집니다.
>
> **SessionManager**를 사용하면:
> - 대화를 **영구 저장**할 수 있고
> - **session_id**로 서로 다른 대화를 **분리**할 수 있습니다.
>
> 아래 실습에서 두 개의 세션을 만들어 각각 다른 대화를 진행해봅니다.

In [10]:
from strands import Agent
from strands.models import BedrockModel
from strands.session.file_session_manager import FileSessionManager

model = BedrockModel(
  model_id="us.amazon.nova-2-lite-v1:0",
  region_name="us-east-1"
)

# 세션 A: 민수의 대화
session_a = FileSessionManager(session_id="user-a", storage_dir="./sessions")
agent_a = Agent(model=model, session_manager=session_a, system_prompt="한국어로 간결하게 답변해. 답변은 2문장 이내로.")

# 세션 B: 영희의 대화
session_b = FileSessionManager(session_id="user-b", storage_dir="./sessions")
agent_b = Agent(model=model, session_manager=session_b, system_prompt="한국어로 간결하게 답변해. 답변은 2문장 이내로.")

print("✅ 두 개의 세션이 생성되었습니다: user-a, user-b")

✅ 두 개의 세션이 생성되었습니다: user-a, user-b


In [11]:
# 세션 A에서 대화
agent_a("내 이름은 민수야. 나는 개발자야.")
print(f"\n[세션 A] 메시지 수: {len(agent_a.messages)}")

민수야, 안녕하세요! 개발자로 활동하고 계시는군요. 꾸준한 학습과 건강 관리를 통해 더 성장하시길 바랍니다.
[세션 A] 메시지 수: 4


In [12]:
# 세션 B에서 대화
agent_b("내 이름은 영희야. 나는 디자이너야.")
print(f"\n[세션 B] 메시지 수: {len(agent_b.messages)}")

영희야, 디자이너로 일을 하는구나! 디자이너로서 창의적인 아이디어를 발휘하며 멋진 작품들을 만들어 나가면 좋을 것 같아.
[세션 B] 메시지 수: 2


In [13]:
# 각 세션이 독립적인지 확인
print("--- 세션 A에게 질문 ---")
agent_a("내 이름이 뭐야?")

print("\n\n--- 세션 B에게 질문 ---")
agent_b("내 이름이 뭐야?")

--- 세션 A에게 질문 ---
민수야, 당신이 말씀하신 이름은 **민수**입니다.  
개발자로 활동하고 계시는군요! 꾸준한 학습과 건강 관리를 통해 더 성장하시길 바랍니다.

--- 세션 B에게 질문 ---
안녕하세요! 사용자님의 이름은 영희입니다. 디자이너로 활동하고 계시는군요.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': '안녕하세요! 사용자님의 이름은 영희입니다. 디자이너로 활동하고 계시는군요.'}], 'metadata': {'usage': {'inputTokens': 140, 'outputTokens': 28, 'totalTokens': 168}, 'metrics': {'latencyMs': 451, 'timeToFirstByteMs': 527}}}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={}, cycle_durations=[1.3197638988494873, 0.9248640537261963], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='602b6121-21a7-49f0-82de-706d4594db02', usage={'inputTokens': 81, 'outputTokens': 50, 'totalTokens': 131})], usage={'inputTokens': 81, 'outputTokens': 50, 'totalTokens': 131}), AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='f4ab500e-5a9b-4e91-9215-30bc757325a4', usage={'inputTokens': 140, 'outputTokens': 28, 'totalTokens': 168})], usage={'inputTokens': 140, 'outputTokens': 28, 'totalTokens': 168})], traces=[<strands.telemetry.metrics.Trace object at 0x11527b7d0>, <strands.telemetry.metrics.Trace obje

In [14]:
# 세션 복원 테스트 — 새 Agent를 같은 session_id로 생성하면 이전 대화가 복원됩니다
restored_session = FileSessionManager(session_id="user-a", storage_dir="./sessions")
restored_agent = Agent(model=model, session_manager=restored_session, system_prompt="한국어로 간결하게 답변해.")

print(f"복원된 메시지 수: {len(restored_agent.messages)}")
print("\n--- 복원된 세션에게 질문 ---")
restored_agent("내 이름이랑 직업이 뭐라고 했지?")

복원된 메시지 수: 6

--- 복원된 세션에게 질문 ---
**민수**야, 당신은 **개발자**라고 했어요!  
계속해서 성장하고 행복한 개발 생활을 하시길 바랍니다. 💻🚀

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': '**민수**야, 당신은 **개발자**라고 했어요!  \n계속해서 성장하고 행복한 개발 생활을 하시길 바랍니다. 💻🚀'}], 'metadata': {'usage': {'inputTokens': 916, 'outputTokens': 44, 'totalTokens': 960}, 'metrics': {'latencyMs': 624, 'timeToFirstByteMs': 617}}}, metrics=EventLoopMetrics(cycle_count=1, tool_metrics={}, cycle_durations=[1.1238961219787598], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='50d37f08-f710-4d33-88d3-f08c05c237f4', usage={'inputTokens': 916, 'outputTokens': 44, 'totalTokens': 960})], usage={'inputTokens': 916, 'outputTokens': 44, 'totalTokens': 960})], traces=[<strands.telemetry.metrics.Trace object at 0x1163440f0>], accumulated_usage={'inputTokens': 916, 'outputTokens': 44, 'totalTokens': 960}, accumulated_metrics={'latencyMs': 624}), state={}, interrupts=None, structured_output=None)

---
## 7. Agent를 AgentCore에 배포하기

> 지금까지 로컬 노트북에서 Agent를 실행했습니다.  
> 이제 이 Agent를 **Amazon Bedrock AgentCore**에 배포하여 API로 호출할 수 있게 만들어봅니다.
>
> ### 배포 흐름
> ```
> 1. agentcore.py 작성 (Agent 코드 + 엔트리포인트)
> 2. 로컬에서 테스트 (python agentcore.py)
> 3. Docker 이미지 빌드 → ECR 업로드
> 4. AgentCore 콘솔에서 배포
> 5. SDK로 호출
> ```
>
> ⚠️ 이미지 빌드/업로드는 시간이 걸리므로, **미리 준비된 공용 ECR 이미지**를 사용합니다.

### AgentCore 콘솔에서 배포

1. [AgentCore 콘솔](https://console.aws.amazon.com/bedrock-agentcore/) 접속
2. **Runtime** → **Create** 클릭
3. 이미지 URI 입력:
   ```
   public.ecr.aws/xxxxxxx/workshop-agent:latest
   ```
4. 배포 완료 후 **Runtime ARN** 복사

> 📋 제공하는 이미지 URI와 Runtime ARN을 사용하세요.

### 배포된 Agent 호출하기

In [13]:
import boto3
import json
import uuid

AWS_REGION='ap-northeast-2'
AGENTCORE_RUNTIME_ARN = 'arn:aws:bedrock-agentcore:ap-northeast-2:055332451319:runtime/hosted_agent_cezm0-f8kINe2Sf0'

client = boto3.client("bedrock-agentcore", region_name=AWS_REGION)

# 세션 ID 생성 (33자 이상 필수)
session_id = "session-" + uuid.uuid4().hex[:33]


response = client.invoke_agent_runtime(
    agentRuntimeArn=AGENTCORE_RUNTIME_ARN,
    runtimeSessionId=session_id, # Must be 33+ char. Every new SessionId will create a new MicroVM
    payload=json.dumps({"prompt": "2026년 5월 5일 일정 알려줘"}).encode()
)

# 응답 출력
response_body = response['response'].read()
response_data = json.loads(response_body)
print("Agent Response:", response_data)

Agent Response: 2026년 5월 5일의 일정은 다음과 같습니다:

- 10:00 팀 미팅
- 14:00 고객사 미팅
- 16:00 코드 리뷰



In [16]:
# 같은 세션으로 추가 질문 (대화 유지)
response = client.invoke_agent_runtime(
    agentRuntimeArn=AGENTCORE_RUNTIME_ARN,
    runtimeSessionId=session_id,  # 같은 세션 ID → 같은 microVM
    payload=json.dumps({"prompt": "그 중에서 가장 중요한 미팅은?"}).encode()
)

response_body = response['response'].read()
response_data = json.loads(response_body)
print("Agent Response:", response_data)

Agent Response: **고객사 미팅(14:00)**이 가장 중요하다고 볼 수 있습니다. 

**이유:**
- **고객과의 관계 강화**: 고객사와 직접 만나고 의견을 나누는 것은 비즈니스 관계를 개선하고 향후 협업에 긍정적인 영향을 미칩니다.
- **프로젝트 방향성 결정**: 고객의 요구사항이나 피드백을 직접 듣고 프로젝트의 방향을 조정할 기회입니다.
- **비즈니스 기회 확대**: 추가 계약이나 협업 기회를 발굴할 수 있는 자리입니다.

다만, 팀 미팅(10:00)이나 코드 리뷰(16:00)도 각각의 목적에 따라 중요할 수 있습니다.  
**구체적인 상황이나 우선순위 기준**이 있다면 추가로 알려주시면 더 정확한 판단을 도와드릴게요!



In [18]:
# 다른 세션으로 추가 질문 (대화 단절)

session_id = "session-" + uuid.uuid4().hex[:33]

response = client.invoke_agent_runtime(
    agentRuntimeArn=AGENTCORE_RUNTIME_ARN,
    runtimeSessionId=session_id,  # 다른 세션 ID → 다른 microVM
    payload=json.dumps({"prompt": "그 중에서 가장 중요한 미팅은?"}).encode()
)

response_body = response['response'].read()
response_data = json.loads(response_body)
print("Agent Response:", response_data)

Agent Response: 일정을 확인하기 위해서는 특정 날짜가 필요합니다. 어떤 날짜의 일정을 확인하고 싶으신가요? 

예를 들어, 오늘 날짜의 일정을 보고 싶으시면 말씀해 주세요. 그럼 현재 시간을 확인하여 오늘의 일정을 조회해 드리겠습니다.

